# Visualisation du corpus OCR

Ce notebook inspecte les manifests du Bulletin officiel : tailles d'images, longueurs des transcriptions, fréquence des caractères et exemples visuels. Il ne modifie pas les données.

In [ ]:
from pathlib import Path
import json
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

def locate_project():
    candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent, Path.cwd().parent.parent.parent]
    for root in candidates:
        if (root / 'outputs' / 'arabic_ocr').is_dir() and (root / 'work' / 'data' / 'gazette').is_dir():
            return root
    raise FileNotFoundError('Projet introuvable : lancez Jupyter depuis le projet ou outputs/arabic_ocr.')

PROJECT = locate_project()
DATA = PROJECT / 'work' / 'data' / 'gazette'
print('Projet :', PROJECT)
print('Données :', DATA)

In [ ]:
def read_manifest(name):
    path = DATA / f'{name}.jsonl'
    rows = [json.loads(line) for line in path.read_text(encoding='utf-8').splitlines() if line.strip()]
    for row in rows:
        row['split'] = name
        row['text_length'] = len(row['text'])
        row['word_count'] = len(row['text'].split())
    return rows

rows = sum((read_manifest(split) for split in ('train', 'val', 'test')), [])
df = pd.DataFrame(rows)
df.groupby('split').agg(samples=('id','size'), mean_chars=('text_length','mean'), median_chars=('text_length','median'), mean_words=('word_count','mean'))

In [ ]:
# Mesurer les dimensions originales des images. Cette cellule peut prendre quelques secondes.
sizes = []
for path in df['image']:
    with Image.open(path) as image:
        sizes.append((image.width, image.height, image.mode))
df[['width_px', 'height_px', 'mode']] = pd.DataFrame(sizes, index=df.index)
df.groupby('split')[['width_px','height_px','text_length']].describe().round(1)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for split, group in df.groupby('split'):
    axes[0].hist(group['text_length'], bins=40, alpha=.55, label=split)
    axes[1].hist(group['width_px'], bins=40, alpha=.55, label=split)
    axes[2].hist(group['height_px'], bins=30, alpha=.55, label=split)
axes[0].set(title='Longueur du texte', xlabel='caractères', ylabel='lignes')
axes[1].set(title='Largeur originale', xlabel='pixels')
axes[2].set(title='Hauteur originale', xlabel='pixels')
for axis in axes: axis.legend()
plt.tight_layout()
plt.show()

In [ ]:
chars = Counter(''.join(df.loc[df['split']=='train', 'text']))
top = pd.DataFrame(chars.most_common(30), columns=['character','count'])
top['label'] = top['character'].replace({' ': '␠', '\n': '↵'})
plt.figure(figsize=(14, 4))
plt.bar(top['label'], top['count'], color='#1764a0')
plt.title('30 caractères les plus fréquents dans train')
plt.ylabel('occurrences')
plt.xticks(rotation=60, ha='right')
plt.tight_layout()
plt.show()
top[['character','count']]

In [ ]:
def show_examples(split, n=8, seed=42):
    sample = df[df['split'] == split].sample(min(n, (df['split'] == split).sum()), random_state=seed)
    fig, axes = plt.subplots(len(sample), 1, figsize=(16, max(2, 2.1*len(sample))))
    axes = np.atleast_1d(axes)
    for axis, (_, row) in zip(axes, sample.iterrows()):
        with Image.open(row['image']) as image:
            axis.imshow(image.convert('L'), cmap='gray')
        axis.set_title(row['text'][:180], loc='left', fontsize=9)
        axis.axis('off')
    fig.suptitle(f'Exemples {split}', y=1.01)
    plt.tight_layout()
    plt.show()

show_examples('train')
show_examples('test')